### Silver Layer (DLT)
Flattens and reformats the bronze payloads into typed, analysis-ready columns..

In [0]:
import dlt
from pyspark.sql import functions as F

In [0]:
@dlt.table(
    name="silver_quotes",
    comment="Cleaned, flattened stock quote snapshots.",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("positive_price", "price > 0")
@dlt.expect_or_drop("valid_volume", "volume >= 0")
@dlt.expect_or_drop("valid_trading_date", "trading_date IS NOT NULL")
def silver_quotes():
    bronze = dlt.read_stream("bronze_quotes")
    return (
        bronze.select(
            F.col("global_quote.`01. symbol`").alias("symbol"),
            F.col("global_quote.`05. price`").cast("double").alias("price"),
            F.col("global_quote.`02. open`").cast("double").alias("open_price"),
            F.col("global_quote.`03. high`").cast("double").alias("day_high"),
            F.col("global_quote.`04. low`").cast("double").alias("day_low"),
            F.col("global_quote.`06. volume`").cast("long").alias("volume"),
            F.to_date(F.col("global_quote.`07. latest trading day`")).alias("trading_date"),
            F.col("global_quote.`08. previous close`").cast("double").alias("previous_close"),
            F.col("global_quote.`09. change`").cast("double").alias("day_change"),
            F.regexp_replace(F.col("global_quote.`10. change percent`"), "%", "")
                .cast("double").alias("day_change_pct"),
            "ingested_at"
        )
        .dropDuplicates(["symbol", "trading_date"])
    )

In [0]:
@dlt.view(name="company_info_flattened")
def company_info_flattened():
    bronze = dlt.read_stream("bronze_company_info")
    return (
        bronze.select(
            F.col("Symbol").alias("symbol"),
            F.col("Name").alias("company_name"),
            F.col("Sector").alias("sector"),
            F.col("Industry").alias("industry"),
            F.col("Exchange").alias("exchange"),
            F.col("Currency").alias("currency"),
            F.col("Country").alias("country"),
            F.col("MarketCapitalization").cast("long").alias("market_cap"),
            F.col("PERatio").cast("double").alias("pe_ratio"),
            F.col("DividendYield").cast("double").alias("dividend_yield"),
            F.col("`52WeekHigh`").cast("double").alias("week_52_high"),
            F.col("`52WeekLow`").cast("double").alias("week_52_low"),
            "ingested_at"
        )
    )

In [0]:
dlt.create_streaming_table(
    name="silver_company_info",
    comment="Company profile attributes with full SCD Type 2 history of changes over time.",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="silver_company_info",
    source="company_info_flattened",
    keys=["symbol"],
    sequence_by=F.col("ingested_at"),
    stored_as_scd_type=2
)